In [ ]:
'''
Here we tried to use Moran model to reanalyze our results got from our previous WF model. During my 3rd committee meeting, Alex
pointed out that the stair-like fitness dynamics we found in facultative sex is an artifact of WF model, as all the germ mutations
get exposed at the same time which greatly reduced the population fitness. Then selection will only work on the individuals with
very low fitness (as they just had sex). 

Here to investigate whether this artifact will matter our results, we tried to use Moran model to re-study this question. This 
new Moran model is different from Joe's previous one. In the new Moran model, the population will adpot a selection-reproduction
-mutation processes in each step (generally within a population with size N, it will take N steps to complete one generation). 

(1) The selection process is same as our WF model, which is achieved by selection with replacement based on their relative fitness.
In each selection, we only select one parent.

(2) For the reproduction process, we introduced another paramter, sex_prob. We use a binomial trial with sex_prob as occurrence 
probability to determine whether sex can occur. If sex cannot occur, then we will let the only parent selected to conduct asexual
reproduction (by either mitosis or amitosis). If sex occur, we will pick another parent randomly based on its relative fitness 
and let it reproduce sexually with the first-selected parent. No matter whether having sex, only one offspring generated will be 
maintained, and replace another randomly selected individual ("dead one") within the population. As the maximum frequency for 
Tetrahymena to have sex is around every 100 generations, the sex_prob is set to be 0.01.

(3) For the mutation process, we will only mutate the maintained offspring generated in reproduction process.

At the same time, we monitored the age of individuals within the population. Once the selected parents participate in reproduction,
their age will be added by 1. For the offspring generated, if it is generated by sex, then the age of offspring will be set to 0.
Otherwise, the age of offspring will be the age of its parents plus 1. Here the age is monitored by the number of steps, dividing
population size N will get the age shown by generation. But in analyses, we mainly foucused on fitness dynamics but didn't focus 
on the age data.

In this code, this is assuming all mutations are deleterious, and the asexual reproduction is conducted by amitosis.


'''

In [ ]:
from __future__ import division
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle

In [ ]:
class Population(object):

    
    def __init__(self,nInds,nLoci,ploidy=45,genomic_mu=0.1,selcoef=0.1, sex_prob = 0.01):

        self.nInds = nInds
        self.nLoci = nLoci
  
        self.soma = np.zeros((nInds,nLoci),dtype='int')
        self.germ = np.zeros((nInds,nLoci),dtype='int')
        self.ploidy = ploidy
        self.mu = genomic_mu/(ploidy*nLoci)
        self.selcoef = selcoef

        self.pop_age = np.zeros(self.nInds, dtype = 'int')

        self.total_sex_age = []
        self.total_dead_age = []
        self.total_sex_trial = []

        self.sex_prob = sex_prob
        self.current_step = 0




    def fitness(self):
        """return a numpy array containing the fitness for each individual in each population"""
        ws = (self.soma.astype('float')/self.ploidy)*self.selcoef  

        each_locus = 1-ws
        fitnesses = np.prod(each_locus, axis =1)

        return fitnesses


            
    def relative_fitness(self):
        """return a numpy array containing each individual's relative fitness"""
        w = self.fitness()
        total_w = np.sum(w)
        totalw = np.expand_dims(total_w,axis=0)
        relfit = w/totalw
        return relfit
                

        

    # Moran model
    
    def pick_parent(self):
        """Randomly choose one parent per population to produce offspring for the next
        time step. Each individual's probability of being chosen is weighted by its 
        relative fitness.
        Moran model method."""
        relfit = self.relative_fitness()
        csrelfit = np.cumsum(relfit,axis=0)
        randvals = np.random.random(1)
        parents = np.searchsorted(csrelfit,randvals)[0]
        return parents
    


    def moran_step(self):

        parent1 = self.pick_parent()
        self.pop_age[parent1] += 1

        self.sex_trial = np.random.binomial(1, self.sex_prob)
        self.total_sex_trial.append(self.sex_trial)

        sex_age = []
        dead_age = []

        if self.sex_trial == 1:

            parent2 = self.pick_parent()
            self.pop_age[parent2]+=1

            sex_age.append([self.pop_age[parent1].copy(), self.pop_age[parent2].copy()])

            
            offspring = self.have_sex(parent1, parent2)
            mut_offspring = self.mutate(offspring)


        else:

            offspring = self.amitosis(parent1)
            mut_offspring = self.mutate(offspring)

        dead = np.random.randint(0,self.nInds)
        dead_age.append(self.pop_age[dead].copy())

        self.total_sex_age.append(sex_age)
        self.total_dead_age.append(dead_age)


        self.soma[dead] = mut_offspring[0]
        self.germ[dead] = mut_offspring[1]


        if self.sex_trial ==1:
            self.pop_age[dead] =0
        else:
            self.pop_age[dead] = self.pop_age[parent1].copy()
                

        self.current_step+=1 







    def mutate(self,individual):


        ind_soma = individual[0]
        ind_germ = individual[1]

        wt_p_soma = self.ploidy - ind_soma
        wt_p_germ = 2 - ind_germ
        soma_mutations = np.random.binomial(wt_p_soma,self.mu)
        germ_mutations = np.random.binomial(wt_p_germ,self.mu)
        ind_soma += soma_mutations
        ind_germ += germ_mutations
        return ind_soma, ind_germ



    def amitosis(self, parent):

        good = (self.ploidy-self.soma[parent])*2
        bad = self.soma[parent]*2

        new_soma = np.random.hypergeometric(bad,good,self.ploidy)
        new_germ = self.germ[parent].copy()

        return new_soma, new_germ



    def have_sex(self, parent1, parent2):

        # First need to generate gametes from MIC
        gamete1 = np.random.hypergeometric(self.germ[parent1],2-self.germ[parent1],1)
        gamete2 = np.random.hypergeometric(self.germ[parent2],2-self.germ[parent2],1)

        # Then two gametes get fused into a diploid zygotes
        zygote = gamete1 + gamete2

        # Then develop into new individual

        new_germ = zygote.copy()

        new_soma = np.zeros(self.nLoci, dtype = 'int')
        new_soma[zygote == 0] =0
        new_soma[zygote == 2] = self.ploidy

        hetcount = len(zygote[zygote==1])
        hetvals = int(self.ploidy/2) + np.random.binomial(self.ploidy-2*int(self.ploidy/2),0.5, hetcount)

        new_soma[zygote ==1] = hetvals

        return new_soma, new_germ



                    

    def get_results(self):

        W = self.fitness()

        W_mean = np.nanmean(W)
        W_var = np.var(W)

        pop_age_mean = np.nanmean(self.pop_age)
        pop_age_var = np.var(self.pop_age)


        return [W_mean, W_var, pop_age_mean, pop_age_var]


        
    def simulate(self,stepcount, strides=10):
 
        results = [self.get_results()]
        
        start = time.time()

        
        while self.current_step <= stepcount:
            self.moran_step()
            if self.current_step%strides == 0:
                results.append(self.get_results())


        end = time.time()
        print 'TOTAL TIME: ',end-start
        return results


def save_object(obj, filename):
    with open(filename, 'wb') as output:
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL)
        


def repeat_simulation(outfile, obj1, obj2, obj3, obj4,  \
    nReps, nInds,nLoci,ploidy=45,genomic_mu=0.1,selcoef=0.1, sex_prob = 0.01, stepcount =2000, strides =10):

    start = time.time()
    total_result = []

    whole_pop_age = []
    whole_total_sex_age = []
    whole_total_dead_age = []
    whole_total_sex_trial = []

    
    for i in range(nReps):
        p = Population(nInds,nLoci,ploidy,genomic_mu,selcoef, sex_prob)    

        result = p.simulate(stepcount, strides)

        total_result.append(result)

        whole_pop_age.append(p.pop_age)
        whole_total_sex_age.append(p.total_sex_age)
        whole_total_dead_age.append(p.total_dead_age)
        whole_total_sex_trial.append(p.total_sex_trial)     
        

    save_object(whole_pop_age, obj1)
    save_object(whole_total_sex_age, obj2)
    save_object(whole_total_dead_age, obj3)
    save_object(whole_total_sex_trial, obj4)


    total_fit = []
    total_var = []
    total_pop_mean_age = []
    total_pop_var_age = []

    for i in range(len(total_result[0])):
        fit = []
        var = []
        pop_mean_age = []
        pop_var_age = []
        
        for j in range(nReps):
            fit.append(total_result[j][i][0])
            var.append(total_result[j][i][1])

            pop_mean_age.append(total_result[j][i][2])
            pop_var_age.append(total_result[j][i][3])

    
        total_fit.append(fit)
        total_var.append(var)
        total_pop_mean_age.append(pop_mean_age)
        total_pop_var_age.append(pop_var_age)



    total_fit = np.array(total_fit)
    total_var = np.array(total_var)
    total_pop_mean_age = np.array(total_pop_mean_age)
    total_pop_var_age = np.array(total_pop_var_age)


    fit_mean = np.nanmean(total_fit, axis =1)
    fit_std = np.nanstd(total_fit, axis =1)
    
    var_mean = np.nanmean(total_var, axis =1)
    var_std = np.nanstd(total_var, axis =1)

    pop_mean_age_mean = np.nanmean(total_pop_mean_age, axis =1)
    pop_mean_age_std = np.nanstd(total_pop_mean_age, axis =1)

    pop_var_age_mean = np.nanmean(total_pop_var_age, axis =1)
    pop_var_age_std = np.nanstd(total_pop_var_age, axis =1)

    
    total = []
    for i in range(len(total_fit)):
        total.append([fit_mean[i], fit_std[i], var_mean[i], var_std[i], \
            pop_mean_age_mean[i],pop_mean_age_std[i], \
            pop_var_age_mean[i], pop_var_age_std[i]])

    colnames = ['Fit_Mean','Fit_STD', 'Fit_Var_Mean', 'Fit_Var_STD', 'Pop_Mean_Age_Mean', 'Pop_Mean_Age_Std', \
    'Pop_Var_Age_Mean', 'Pop_Var_Age_Std']

    results = pd.DataFrame(np.array(total),columns=colnames)
    results.to_csv(outfile,index=False)
    
    end = time.time()
    print 'TOTAL TIME: ',end-start

In [ ]:
repeat_simulation('Fit_RM_DeleOnly_1116.csv', 'RM_DeleOnly_Final_PopAge', 'RM_DeleOnly_TotalSex_Age', 'RM_DeleOnly_TotalDead_Age', 'RM_DeleOnly_TotalSex_Trial', \
    nReps =500, nInds =20,nLoci =100,ploidy=45,genomic_mu=0.1,selcoef=0.1, sex_prob = 1, stepcount =2000*20, strides =20)